<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/03_heavy_tailed_slope_prior.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 3 — Can heavy tails make a skeptical prior more robust?

In Notebook 2, a tight Normal prior on the population slope conflicted with information in the data.

Here we keep the same skeptical center and scale but replace the Normal prior with a heavier-tailed Student-t prior. The goal is to see whether changing only the tail behavior makes the prior more robust to data that favor a much larger effect.

## Setup

This course pins PyMC and the modular ArviZ packages for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from scipy import stats
from pymc.stats.log_density import compute_log_density
import pymc as pm
import arviz_base as azb
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. You can read about the original study here: [Belenky J Sleep Res. 2003](https://doi.org/10.1046/j.1365-2869.2003.00337.x)

Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

That zero point is scientifically meaningful, so every model in this sequence keeps `Days` on its natural scale (no centering).

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

print(f"{sleep['Subject'].nunique()} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

### Plotting helper

We will use the same participant-level predictive plot as in the previous notebooks.

In [ ]:
LM_VISUALS = {
    "pe_line": {"color": "C1"},
    "ci_band": {"color": "C0"},
    "observed_scatter": {"color": "black", "alpha": 1, "zorder": 3, "s": 10},
}

PARTICIPANT_DAY = xr.Coordinates.from_pandas_multiindex(
    pd.MultiIndex.from_frame(
        sleep[["Subject", "Days"]],
        names=["participant", "day"],
    ),
    "obs_id",
)

def plot_participants(dt, group, var):
    """One panel per participant: bands for `var` against days, with observed data."""
    def reshape(ds):
        return ds.assign_coords(PARTICIPANT_DAY).unstack("obs_id")

    panels = xr.DataTree.from_dict({
        group: reshape(dt[group].to_dataset()),
        "observed_data": reshape(dt["observed_data"].to_dataset()),
        "constant_data": reshape(dt["constant_data"].to_dataset()),
    })

    pc = azp.plot_lm(
        panels,
        x="days",
        y=var,
        y_obs="y",
        group=group,
        plot_dim="day",
        ci_prob=(0.50, 0.90),
        ci_kind="hdi",
        point_estimate="mean",
        smooth=False,
        cols=["participant"],
        col_wrap=6,
        figure_kwargs={"figsize": (11, 5.5), "sharex": True, "sharey": True},
        visuals={**LM_VISUALS, "xlabel": False, "ylabel": False},
    )

    pc.add_legend("prob", title="HDI")

    fig = pc.get_viz("figure")
    fig.supxlabel("Days of sleep deprivation")
    fig.supylabel("Reaction time (ms)")

    return pc

## 1. Change the tail behavior, not the skeptical center

### 1.1 How can we keep a skeptical prior but make it less brittle?

Notebook 2 concentrated the slope prior tightly around zero. How can we preserve that skepticism near zero while allowing rare, much larger slopes?

We can replace the Normal prior with a **heavy-tailed Student-t prior** that has the same center and scale.

We will use

$$
b_1 \sim \operatorname{StudentT}(\nu=7,\mu=0,\sigma=1).
$$

Near zero this resembles the `Normal(0, 1)` prior from Notebook 2, but its heavier tails assign substantially more probability to large positive or negative slopes.

### 1.2 How does the Student-t prior differ from the Normal prior?

The next cell plots the two prior densities using the same center and scale.

In [ ]:
x = np.linspace(-6, 6, 800)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(x, stats.norm.pdf(x, 0, 1), label="Normal(0, 1)")
ax.plot(x, stats.t.pdf(x, df=7), label="Student-t(7, 0, 1)")
ax.set(xlabel="Daily effect (ms/day)", ylabel="Density")
ax.legend(frameon=False)
plt.show()

Where is the difference between the two priors most important?

- answer here

### 1.3 Build the model with the heavy-tailed slope prior.

Use the same data, likelihood, and priors for `b0` and `sd_y` as before. Use `pm.StudentT` for `b1`.

In [ ]:
mu_b0 = 250
sd_b0 = 100

mu_b1 = 0
sd_b1 = 1
nu_b1 = 7

mu_sd_y = 50

In [ ]:
# answer here

## 2. Prior predictive check — Does the heavy-tailed prior look acceptable before fitting?

### 2.1 Generate prior predictive reaction times.

Draw 500 samples from the prior predictive distribution for `y`.

In [ ]:
# answer here

### 2.2 Plot the prior predictive reaction times.

Use the supplied `plot_participants` function.

In [ ]:
# answer here

### 2.3 Do the prior predictive simulations meet the criteria established in Notebook 1?

Apply the same criteria used in the previous notebooks. In particular, does the prior put enough probability on substantial changes across the seven days without routinely generating absurd trajectories?

- answer here

### 2.4 Does the prior predictive plot look very different from Notebook 2?

- answer here

## 3. Fit and diagnose — Can the model sample reliably?

### 3.1 Fit the model.

Use the same sampling settings as before: 1000 posterior draws in each of 4 chains after 1500 tuning draws.

In [ ]:
# answer here

### 3.2 Check the sampling diagnostics.

Count divergences, summarize `b0`, `b1`, and `sd_y`, and inspect their trace/distribution plots using the criteria established in Notebook 1.

In [ ]:
# answer here

In [ ]:
# answer here

### 3.3 Did sampling succeed?

Do the numerical and graphical diagnostics indicate that the posterior samples are computationally reliable?

- answer here

## 4. Posterior predictive check — Did the heavier tails improve model fit?

### 4.1 Generate and plot posterior predictive reaction times.

Generate replicated values of `y` from the fitted model and compare them with the observations using `plot_participants`.

In [ ]:
# answer here

### 4.2 Does the fitted model now meet the posterior predictive criteria from Notebook 1?

- answer here

### 4.3 Compared with Notebook 2, did changing only the prior tails improve the model's ability to reproduce the systematic increase across days?

- answer here

## 5. Prior sensitivity — Did the heavy tails eliminate the prior–data conflict?

### 5.1 Why check prior sensitivity if the posterior predictive fit improved?

Improved prediction does not tell us whether the prior and likelihood are now in agreement.

Power-scaling sensitivity lets us ask whether small changes in the weight of the prior or likelihood still move the posterior substantially. That directly tests whether the prior–data conflict seen in Notebook 2 has actually been resolved.

### 5.2 Prepare the information required for power-scaling.

The bookkeeping code is supplied.

In [ ]:
with model:
    pm.compute_log_likelihood(idata)
    compute_log_density(
        idata,
        model=model,
        kind="prior",
        extend_inferencedata=True,
    )

### 5.3 Calculate and visualize prior sensitivity.

Use `azs.psense_summary` and `azp.plot_psense_dist` for `b0`, `b1`, and `sd_y`.

In [ ]:
# answer here

In [ ]:
# answer here

### 5.4 Did the heavy-tailed prior eliminate the prior–data conflict?

Does the sensitivity analysis still flag a problem?

- answer here